# Une demande déplaçable en trente lignes

*Ajouter une FlexibleDemand au modèle du tutoriel, vérifier que chaque fenêtre conserve son volume, lire les heures qu'elle quitte et celles qu'elle rejoint, et voir ce que fait le gain quand la part déplaçable augmente chez un preneur de prix.*

Ce notebook est tiré du texte [en ligne](https://www.robingirard.eu/assets/parcours-es/latest/textes/act-demande-flexible-craft.html) du parcours ES ; exécutez les cellules dans l'ordre (Maj + Entrée). Les commentaires des cellules donnent les valeurs attendues.

**La cellule d'installation.** En local, placez ce notebook dans le dépôt `pommes-case-studies` (à sa racine ou dans un de ses dossiers) et ouvrez-le avec le noyau du venv du dépôt (`.venv`), comme l'explique la page [Installer et vérifier](https://www.robingirard.eu/assets/parcours-es/latest/textes/installer.html) : la cellule n'installe rien, elle trouve le dépôt et affiche les versions des paquets. Une ligne « Attention » signale un autre noyau : changez-le (Kernel › Change Kernel).

Sur Google Colab (Fichier › Importer un notebook), la cellule clone le dépôt et installe les paquets : 2 à 3 minutes, puis elle s'arrête volontairement sur `RedemarrezLaSession`. Faites Exécution › Redémarrer la session, relancez cette cellule, et seulement elle, puis continuez dans l'ordre.

In [ ]:
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

VERSIONS = {"pommes": "0.11.0", "pommes-craft": "0.6.0", "linopy": "0.10.1", "highspy": "1.15.1"}


def versions_du_depot() -> bool:
    """Vrai si les paquets installés sont ceux de requirements.txt (les valeurs attendues en dépendent)."""
    try:
        return all(version(nom) == v for nom, v in VERSIONS.items())
    except PackageNotFoundError:
        return False


class RedemarrezLaSession(Exception):
    """Colab : les paquets sont installés, mais la session garde en mémoire les anciens numpy et pandas."""


if "google.colab" in sys.modules:                     # sur Colab : cloner, installer, redémarrer une fois
    CLONE = Path("/content/pommes-case-studies")
    if not CLONE.exists():
        !git clone https://git.persee.minesparis.psl.eu/energy-alternatives/pommes_studies/pommes-case-studies {CLONE}
    %cd {CLONE}
    if not versions_du_depot():
        !python --version
        !pip install -q -r requirements.txt
        raise RedemarrezLaSession("Installation finie. Exécution › Redémarrer la session, "
                                  "puis relancez cette cellule (et seulement elle).")

RACINE = next((d for d in (Path.cwd(), *Path.cwd().parents, Path.cwd() / "pommes-case-studies")
               if (d / "pommes_cases").is_dir()), None)  # la racine du dépôt des cas
if RACINE is None:
    raise FileNotFoundError(
        "dépôt pommes-case-studies introuvable : ce notebook a besoin de son code et de ses données. "
        "Clonez le dépôt, placez ce notebook dedans (à sa racine ou dans un de ses dossiers) et ouvrez-le avec "
        "le noyau du venv du dépôt (https://www.robingirard.eu/assets/parcours-es/latest/textes/installer.html). "
        "Sans rien installer : ouvrez-le sur Google Colab.")
if str(RACINE) not in sys.path:
    sys.path.insert(0, str(RACINE))
print("dépôt :", RACINE.name, "· Python", sys.version.split()[0])
print(" · ".join(f"{nom} {version(nom)}" for nom in VERSIONS))
if not versions_du_depot():
    print("Attention : ce ne sont pas les versions du dépôt", VERSIONS, ": les valeurs attendues peuvent bouger.")
    print("Le noyau n'est sans doute pas celui du venv du dépôt : Kernel › Change Kernel.")

Objectif : rendre déplaçable une part de la demande du modèle du tutoriel [Le premier modèle sur une page : les huit étapes du notebook et leurs valeurs attendues](https://www.robingirard.eu/assets/parcours-es/latest/textes/premier-modele.html) et lire ce que l'optimiseur en fait. Prérequis : le dépôt des cas installé [Installer et vérifier en dix minutes : venv du dépôt des cas, test de contrôle, dépannage](https://www.robingirard.eu/assets/parcours-es/latest/textes/installer.html), le bloc 2 du tutoriel compris, la différence entre preneur et faiseur de prix [Distinguer un acteur preneur de prix d'une flotte qui fait le prix : la cannibalisation du n-ième GW](https://www.robingirard.eu/assets/parcours-es/latest/index.html#c-flex-preneur-vs-faiseur). Chaque résolution prend moins de deux secondes (pommes 0.11.0, pommes_craft 0.6.0, HiGHS) ; valeurs du 16/11/2026, revérifiées le 17/11/2026.

## 1. Le composant

`FlexibleDemand` (`pommes_craft/components/flexibility.py`) prend sept arguments. `demand` est le profil de référence, en MW par heure ; `conservation_hrs`, la longueur de la fenêtre où le volume est conservé. `min_demand` et `max_demand` bornent la demande horaire après déplacement, `ramp_up` et `ramp_down` ses variations d'une heure à l'autre. `variable_cost` est le coût en euros par MWh déplacé. Pommes crée une variable `operation_flexibility_demand` par heure, la met dans l'équilibre à la place du profil et impose sur chaque fenêtre `somme(variable) = somme(profil)`. Le coût ne compte que la demande retirée d'une heure, jamais celle qui s'y ajoute.

## 2. Le modèle en trente lignes

Même squelette que le tutoriel : une zone, un réseau qui vend au prix horaire 2030 de la France (TYNDP 2026, NT+) et ne rachète rien, ni délestage ni écrêtement. La demande de 1 MW est coupée en `1 - part` MW rigides et `part` MW déplaçables dans la journée, plafonnés à quatre fois leur moyenne. `progress: False` tait les barres de progression de linopy ; la ligne `Flexibility demand sparsity` qu'affichait pommes 0.9.0 à chaque résolution, passée au niveau `debug` du journal, ne s'affiche plus. Les commentaires donnent la sortie des `print`.

In [ ]:
import sys, logging
import numpy as np, polars as pl
from pommes_cases.prices import load_prices
from pommes_craft import (EnergyModel, Area, Demand, EconomicHypothesis, TimeStepManager,
                          LoadShedding, Spillage, NetImport, FlexibleDemand)
logging.getLogger().setLevel(logging.ERROR)
p = load_prices("tyndp2026", "NT+", "FR", 2030)
prix = p["import_price"].to_numpy()
H = 8760

def modele(nom, part=0.0, fenetre=24, pmax=None, cout=0.0):
    m = EnergyModel(name=nom, hours=list(range(H)), year_ops=[2030], year_invs=[2030], year_decs=[2055],
                    modes=[], resources=["electricity"], solver_name="highs",
                    solver_options={"output_flag": False, "progress": False})
    with m.context():
        EconomicHypothesis("eco", discount_rate=0.0, year_ref=2030, planning_step=1)
        TimeStepManager("pas", time_step_duration=1.0, operation_year_duration=float(H))
        zone = Area("FR")
        zone.add_component(Demand(name="demande_fixe", resource="electricity",
                                  demand=pl.DataFrame({"hour": range(H), "year_op": 2030, "demand": 1.0 - part})))
        if part > 0:
            zone.add_component(FlexibleDemand(name="demande_flex", resource="electricity",
                                              demand=pl.DataFrame({"hour": range(H), "year_op": 2030, "demand": part}),
                                              conservation_hrs=fenetre, min_demand=0.0,
                                              max_demand=4 * part if pmax is None else pmax,
                                              variable_cost=cout))
        zone.add_component(LoadShedding(name="delestage", resource="electricity", max_capacity=0.0))
        zone.add_component(Spillage(name="ecretement", resource="electricity", max_capacity=0.0))
        zone.add_component(NetImport(name="reseau", resource="electricity",
                                     import_price=p.select("hour", "year_op", "import_price"),
                                     max_yearly_energy_export=0.0))
    return m

m0 = modele("rigide"); m0.run(store_linopy_model=True)
m1 = modele("flex20", part=0.2); m1.run(store_linopy_model=True)
c0, c1 = m0.linopy_model.objective.value, m1.linopy_model.objective.value
print(f"rigide {c0:,.2f} €, 20 % déplaçable {c1:,.2f} €, gain {c0 - c1:,.2f} €")
# rigide 503,709.84 €, 20 % déplaçable 467,376.26 €, gain 36,333.58 €

Le modèle rigide redonne la somme des prix du tutoriel. Un cinquième de demande déplaçable dans la journée fait gagner 36 334 €, soit 7,2 % de la facture.

## 3. Vérifier la conservation, lire le déplacement

In [ ]:
d = m1.linopy_model.solution["operation_flexibility_demand"].squeeze().values
print(d.sum(), np.abs(d.reshape(365, 24).sum(axis=1) - 0.2 * 24).max())   # 1752.0  6.5e-11
hausse, baisse = d > 0.2 + 1e-6, d < 0.2 - 1e-6
print(hausse.sum(), prix[hausse].mean(), baisse.sum(), prix[baisse].mean())  # 2190 36.76  6570 64.41
print(np.maximum(0.2 - d, 0).sum(), d.max())                                 # 1314.0 MWh  0.8 MW
hd = np.arange(H) % 24
print(np.round([d[hd == h].mean() for h in range(24)], 2))   # pic 0.71 MW à 13 h, 0.00 à 8 h
print(prix @ d / d.sum())                                     # 36.76 €/MWh payés, contre 57.50 en rigide
j = 83                                                        # indice 0 : le 84e jour civil
print(np.sort(prix.reshape(365, 24)[j][d.reshape(365, 24)[j] > 0.5]))  # arrondi : 0.01 0.01 0.01 0.03 0.03 15.63

Les 365 fenêtres conservent 4,8 MWh chacune, à 10⁻¹⁰ près ; l'année fait toujours 1 752 MWh. La solution est en tout ou rien : 0,8 MW pendant 2 190 heures, six par jour, 0 le reste du temps. La demande quitte 6 570 heures à 64,41 €/MWh en moyenne pour six heures à 36,76 €/MWh. Le profil moyen se creuse le matin et le soir et culmine à 13 h : en 2030, l'heure creuse est solaire. Le jour d'indice 83 retient six heures à au plus 15,63 €/MWh, dont cinq sous 0,05 ; la septième la moins chère vaut aussi 15,63 : l'optimiseur a tranché l'égalité. La veille, indice 82, en retient six entre 15,62 et 15,63, aucune à 0. La duale de l'équilibre égale le prix à toutes les heures : le réseau reste marginal.

`m1.get_results("operation", "demand", component_classes=[FlexibleDemand])` donne la même série en table (colonnes `hour`, `value`, `datetime`, entre autres).

## 4. Le gain quand la part déplaçable augmente

In [ ]:
def gain(**kw):
    m = modele("x", **kw); m.run(store_linopy_model=True)
    return c0 - m.linopy_model.objective.value
for part in (0.1, 0.2, 0.3, 0.5):
    g = gain(part=part); print(part, round(g), round(g / (part * H), 2))
# 0.1 18167 20.74 | 0.2 36334 20.74 | 0.3 54500 20.74 | 0.5 90834 20.74
for part in (0.1, 0.2, 0.3, 0.5, 0.8):
    g = gain(part=part, pmax=1.0); print(part, round(g), round(g / (part * H), 2))
# 0.1 20610 23.53 | 0.2 38285 21.85 | 0.3 50334 19.15 | 0.5 50994 11.64 | 0.8 28836 4.11

Première série, plafond proportionnel : le gain vaut 20,74 € par MWh déplaçable, quelle que soit la part. Les prix ne bougent pas, chaque MWh de plus trouve les mêmes six heures : signature du preneur de prix. Deuxième série, plafond fixe de 1 MW, celui d'un raccordement : le gain par MWh tombe de 23,53 à 4,11 €, et le gain total plafonne à 51 000 € vers 50 % puis recule. À 0,8 MW déplaçables, écouler le volume demande 19 heures par jour à 1 MW : les heures creuses saturent, le reste part aux heures chères. Chez un preneur de prix, c'est la borne de puissance qui fait baisser le gain, pas le prix. Dans le cas B, c'est le prix qui bouge : la flotte fait fondre les heures sous 10 €/MWh [Cas B — VE pilotés : que deviennent les heures à prix bas ?](https://git.persee.minesparis.psl.eu/energy-alternatives/pommes_studies/pommes-case-studies/-/tree/main/B_ve_pilotes).

## 5. Fenêtre et coût du déplacement

Même part de 20 % : une fenêtre de 4 h rapporte 8 920 €, 12 h 34 225 €, 24 h 36 334 €, une semaine 46 927 €. Le volume déplacé reste 1 314 MWh : c'est l'écart de prix accessible qui grandit. Un coût de déplacement de 10 €/MWh ramène le gain net à 25 973 € et le volume déplacé à 953 MWh ; 20 €/MWh, 17 401 € et 771 MWh ; 40 €/MWh, 6 352 € et 348 MWh. L'optimiseur ne déplace que les heures dont l'écart de prix dépasse le coût.

## Trois pièges

Une heure où le profil de référence vaut zéro n'a pas de variable : la demande ne peut pas s'y déplacer. Prenez 24 heures, 20 €/MWh de 0 à 7 h et de 20 à 23 h, 100 €/MWh de 8 à 19 h, une fenêtre de 24 h et `max_demand=10`. Un profil de 1,5 MW le jour et nul la nuit coûte 1 800 € (12 h × 1,5 MW × 100 €/MWh) : rien ne bouge. Le même profil avec 0,001 MW la nuit coûte 360,24 € : les 18,012 MWh passent tous la nuit, à 20 €/MWh. Donnez un plancher positif à votre profil. Ensuite, `conservation_hrs=0`, le défaut, ne conserve que l'année entière : la demande se concentre sur ses heures les moins chères. Enfin, le coût du déplacement était écrit en une ligne par heure et un terme par heure avant pommes 0.9.0 : plus de 60 Gio sur 13 zones à 8 760 heures, et faux à coût horaire variable (`docs/S1_rapport.md`). Restez sur l'environnement épinglé [Les pièges connus de pommes et pommes_craft : symptôme, cause, remède](https://www.robingirard.eu/assets/parcours-es/latest/textes/pieges-pommes.html).

## L'autre façon : la flotte du cas B

`pommes_cases.systeme_fr.add_ev` écrit la même flexibilité avec trois composants et un garde-fou. Une `Demand` de conduite porte sur une ressource « ve » ; une `ConversionTechnology` de recharge convertit l'électricité en « ve » sous un plafond horaire fixé par les voitures branchées ; une `StorageTechnology` de 16 kWh par voiture s'intercale entre recharge et conduite. Le garde-fou, un `NetImport` de « ve » borné à zéro, empêche pommes d'importer la ressource gratuitement au lieu de la recharger. Cette forme borne la puissance par les véhicules branchés et l'énergie par la batterie, là où `FlexibleDemand` n'a que `max_demand` et une fenêtre ; elle n'a pas de coût du déplacement [Cas B — VE pilotés : que deviennent les heures à prix bas ?](https://git.persee.minesparis.psl.eu/energy-alternatives/pommes_studies/pommes-case-studies/-/tree/main/B_ve_pilotes).

## Trois questions à réponse chiffrée

1. Avec 20 % déplaçables et un plafond à 4 fois la moyenne, combien d'heures par jour la demande déplaçable est-elle active, et à quel prix moyen ? (6 heures, 36,76 €/MWh.)
2. Quel gain par MWh déplaçable obtient-on à 10 %, 30 %, 50 % avec le plafond proportionnel, et pourquoi est-il constant ? (20,74 €/MWh : preneur de prix.)
3. À quel coût de déplacement le volume déplacé tombe-t-il sous 400 MWh ? (40 €/MWh : 348 MWh.)

## Vous l'avez acquise quand…

- Vous retrouvez 467 376,26 € et un écart de conservation par fenêtre inférieur à 10⁻⁸ MWh.
- Vous tracez le profil horaire moyen de la demande déplacée et expliquez le pic de 13 h par le prix.
- Vous prédisez, avant de lancer, lequel des deux plafonds donne un gain par MWh constant.
- Vous dites en une phrase ce que la flotte du cas B borne que `FlexibleDemand` ne borne pas.

## Pour aller plus loin

- Model description de pommes : les contraintes de la demande flexible, rampes comprises [Model description (3) : demande flexible et groupes de production](https://pages.persee.minesparis.psl.eu/energy-alternatives/pommes/methodology/model_description.html).
- Cas B : la même question à l'échelle de 33 millions de voitures, où le prix répond [Cas B — VE pilotés : que deviennent les heures à prix bas ?](https://git.persee.minesparis.psl.eu/energy-alternatives/pommes_studies/pommes-case-studies/-/tree/main/B_ve_pilotes).
- Le tutoriel dont ce modèle est la suite directe [Le premier modèle sur une page : les huit étapes du notebook et leurs valeurs attendues](https://www.robingirard.eu/assets/parcours-es/latest/textes/premier-modele.html).
- Les pièges de pommes et pommes_craft, dont la correction du coût flexible en 0.9.0 [Les pièges connus de pommes et pommes_craft : symptôme, cause, remède](https://www.robingirard.eu/assets/parcours-es/latest/textes/pieges-pommes.html).

## Sources

1. [Dépôt pommes-case-studies (pommes_cases/prices.py, pommes_cases/systeme_fr.py add_ev, docs/S1_rapport.md, B_ve_pilotes/README.md)](https://git.persee.minesparis.psl.eu/energy-alternatives/pommes_studies/pommes-case-studies)
2. [Doc pommes, Model description, section Flexible demand constraints](https://pages.persee.minesparis.psl.eu/energy-alternatives/pommes/methodology/model_description.html)
3. [CHANGELOG de pommes (correction du coût de la demande flexible en 0.9.0)](https://git.persee.minesparis.psl.eu/energy-alternatives/pommes/-/blob/main/CHANGELOG.md)
4. [README de pommes_craft sur PyPI](https://pypi.org/project/pommes-craft/)